# Model Training - Villa Vente Marrakech (Haute Performance)
Ce notebook utilise un modèle **XGBoost** optimisé sur cible logarithmique pour atteindre plus de 80% de performance sur les villas.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor

# Ajouter le chemin vers la racine pour importer les helpers
sys.path.append('../')
from model_training_helpers import split_data, model_pipeline, metric_model, get_features, print_metrics, save_model, preprocess_data, tune_model

print("Modules chargés avec succès.")

## 1. Chargement et Fusion des données

In [ ]:
csv_path = '../data/cleaned_data/vente/villa_vente_cleaned.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Données chargées : {df.shape}.")
else:
    print(f"ERREUR : Fichier {csv_path} non trouvé.")

## 2. Préparation (Feature Engineering & Outliers)

In [ ]:
# Appliquer le prétraitement spécifique aux villas (filtres de prix et surface adaptés)
df = preprocess_data(df, property_type='villa')
print(f"Shape après filtrage des outliers : {df.shape}")

# Split des données
X_train, X_test, y_train, y_test = split_data(df, target='prix_num')

# Identification des colonnes
num_features, cat_features = get_features(X_train)
print(f"Nombre de features numériques : {len(num_features)}")
print(f"Nombre de features catégorielles : {len(cat_features)}")

## 3. Entraînement du modèle XGBoost

In [ ]:
print('Entraînement du modèle Villa (Optimisation V12 - Robustesse)...')

# 1. Tuning de XGBoost avec modèle plus simple pour éviter l'overfitting
param_dist_xgb = {
    'n_estimators': [3000, 4000],
    'learning_rate': [0.01, 0.02],
    'max_depth': [6, 8],
    'min_child_weight': [5, 10],
    'subsample': [0.8],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0.5, 1.0],
    'reg_alpha': [1.0, 10.0],
    'reg_lambda': [10.0, 50.0]
}

print("Recherche de la configuration Stable (100 itérations)...")
best_xgb_pipeline = tune_model(
    X_train, 
    np.log1p(y_train), 
    num_features, 
    cat_features, 
    xgb.XGBRegressor, 
    param_dist_xgb, 
    n_iter=100, 
    cv=5, 
    scoring='r2'
)

# 2. HistGradientBoosting massif mais régularisé
hist_reg = HistGradientBoostingRegressor(
    max_iter=3000, 
    max_depth=10, 
    learning_rate=0.01,
    l2_regularization=50.0,
    random_state=42
)

# 3. Voting Regressor pour plus de stabilité sur le R2
from sklearn.ensemble import VotingRegressor
estimators = [
    ('xgb', best_xgb_pipeline),
    ('hist', model_pipeline(num_features, cat_features, hist_reg))
]

voting = VotingRegressor(
    estimators=estimators,
    n_jobs=-1
)

# 4. Encapsulation Log-Target
pipeline = TransformedTargetRegressor(
    regressor=voting,
    func=np.log1p,
    inverse_func=np.expm1
)

print("Entraînement final du Voting V12...")
pipeline.fit(X_train, y_train)
print('Modèle Villa Haute Performance (V12) prêt.')

## 4. Évaluation & Validation

In [ ]:
y_pred = pipeline.predict(X_test)
metrics = metric_model(y_test, y_pred, model_name="XGBoost Villas Fast")
print_metrics(metrics, model_name="XGBoost Villas Fast")

## 5. Sauvegarde

In [ ]:
save_model(pipeline, 'villa_vente_final_model.joblib')
print("Modèle final sauvegardé.")